In [0]:
import uuid, datetime
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

CATALOG, SCHEMA = "workspace", "github_health"
T = lambda n: f"{CATALOG}.{SCHEMA}.{n}"

# =====================================================================
# 1. EXPLICIT SCHEMAS (StructType / StructField, no inferSchema anywhere)
# =====================================================================
S, L, B, I = StringType(), LongType(), BooleanType(), IntegerType()
user_t   = StructType([StructField("login", S), StructField("id", L), StructField("type", S)])
person_t = StructType([StructField("name", S), StructField("email", S), StructField("date", S)])
meta     = [StructField("_repo", S), StructField("_ingested_at", S)]

SCHEMAS = {
 "commits": StructType([
    StructField("sha", S),
    StructField("commit", StructType([StructField("author", person_t),
                                      StructField("committer", person_t),
                                      StructField("message", S),
                                      StructField("comment_count", L)])),
    StructField("author", user_t), StructField("committer", user_t),
    StructField("parents", ArrayType(StructType([StructField("sha", S)]))),
    StructField("html_url", S)] + meta),

 "issues": StructType([
    StructField("id", L), StructField("number", L), StructField("title", S),
    StructField("state", S), StructField("state_reason", S), StructField("user", user_t),
    StructField("labels", ArrayType(StructType([StructField("name", S)]))),
    StructField("comments", L), StructField("locked", B),
    StructField("author_association", S), StructField("body", S),
    StructField("created_at", S), StructField("updated_at", S), StructField("closed_at", S)] + meta),

 "pulls": StructType([
    StructField("id", L), StructField("number", L), StructField("title", S),
    StructField("state", S), StructField("draft", B), StructField("user", user_t),
    StructField("base", StructType([StructField("ref", S)])),
    StructField("head", StructType([StructField("ref", S)])),
    StructField("created_at", S), StructField("updated_at", S),
    StructField("closed_at", S), StructField("merged_at", S)] + meta),

 "contributors": StructType([
    StructField("login", S), StructField("id", L), StructField("type", S),
    StructField("contributions", L)] + meta),

 "repo_metadata": StructType([
    StructField("id", L), StructField("name", S), StructField("full_name", S),
    StructField("owner", user_t), StructField("description", S), StructField("language", S),
    StructField("license", StructType([StructField("spdx_id", S)])),
    StructField("default_branch", S), StructField("archived", B),
    StructField("stargazers_count", L), StructField("forks_count", L),
    StructField("open_issues_count", L), StructField("size", L),
    StructField("created_at", S), StructField("pushed_at", S)] + meta),
}

# Bronze primary keys (always include _repo where ids are not globally unique)
BRONZE_PK = {
    "commits":       ["sha", "_repo"],
    "issues":        ["id", "_repo"],
    "pulls":         ["id", "_repo"],
    "contributors":  ["id", "_repo", "_ingested_at"],
    "repo_metadata": ["id", "_ingested_at"],
}

# =====================================================================
# 2. SILVER: casting + renaming
# =====================================================================
# try_to_timestamp returns NULL on bad input instead of failing (serverless uses ANSI mode)
tts = lambda c, a: F.expr(f"try_to_timestamp({c})").alias(a)

SILVER = {
 "commits": dict(pk=["sha", "repo"], cols=[
    F.col("sha"), F.col("_repo").alias("repo"),
    F.col("commit.author.name").alias("author_name"),
    F.col("commit.author.email").alias("author_email"),
    tts("commit.author.date", "authored_at"), tts("commit.committer.date", "committed_at"),
    F.col("author.login").alias("author_login"),
    F.size("parents").cast("int").alias("parent_count"),
    F.col("commit.message").alias("message"), F.col("html_url")]),

 "issues": dict(pk=["issue_id", "repo"], cols=[
    F.col("id").alias("issue_id"), F.col("_repo").alias("repo"), F.col("number").alias("issue_number"),
    "title", "state", "state_reason", F.col("user.login").alias("author_login"),
    F.col("author_association"), F.expr("transform(labels, x -> x.name)").alias("labels"),
    F.col("comments").cast("int").alias("comment_count"), F.col("locked").alias("is_locked"),
    tts("created_at", "created_at"), tts("updated_at", "updated_at"), tts("closed_at", "closed_at"),
    "body",
    # derived DOUBLE column
    ((F.unix_timestamp(F.expr("try_to_timestamp(closed_at)")) -
      F.unix_timestamp(F.expr("try_to_timestamp(created_at)"))) / 3600).cast("double").alias("hours_to_close")]),

 "pulls": dict(pk=["pr_id", "repo"], cols=[
    F.col("id").alias("pr_id"), F.col("_repo").alias("repo"), F.col("number").alias("pr_number"),
    "title", "state", F.col("draft").alias("is_draft"), F.col("user.login").alias("author_login"),
    F.col("base.ref").alias("base_branch"), F.col("head.ref").alias("head_branch"),
    tts("created_at", "created_at"), tts("updated_at", "updated_at"),
    tts("closed_at", "closed_at"), tts("merged_at", "merged_at"),
    F.col("merged_at").isNotNull().alias("is_merged"),
    # derived DOUBLE column
    ((F.unix_timestamp(F.expr("try_to_timestamp(merged_at)")) -
      F.unix_timestamp(F.expr("try_to_timestamp(created_at)"))) / 3600).cast("double").alias("hours_to_merge")]),

 "contributors": dict(pk=["repo", "user_id", "snapshot_date"], cols=[
    F.col("_repo").alias("repo"), F.col("id").alias("user_id"), "login", "type",
    F.col("contributions").cast("int").alias("contributions"),
    F.to_date(F.expr("try_to_timestamp(_ingested_at)")).alias("snapshot_date")]),

 "repo_metadata": dict(pk=["repo_id", "snapshot_date"], cols=[
    F.col("id").alias("repo_id"), F.col("full_name").alias("repo"), "name",
    F.col("owner.login").alias("owner_login"), "description", "language",
    F.col("license.spdx_id").alias("license_spdx"), "default_branch",
    F.col("archived").alias("is_archived"),
    F.col("stargazers_count").alias("stars"), F.col("forks_count").alias("forks"),
    F.col("open_issues_count").alias("open_issues"), F.col("size").alias("size_kb"),
    tts("created_at", "created_at"), tts("pushed_at", "pushed_at"),
    F.to_date(F.expr("try_to_timestamp(_ingested_at)")).alias("snapshot_date")]),
}

# =====================================================================
# 3. AUDIT LOGGING
# =====================================================================
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T('pipeline_execution_logs')} (
  batch_id STRING, layer STRING, entity STRING, parameter STRING, load_type STRING,
  start_time TIMESTAMP, end_time TIMESTAMP, status STRING,
  rows_read BIGINT, rows_inserted BIGINT, rows_updated BIGINT, rows_quarantined BIGINT,
  drift_columns STRING, error_message STRING, load_timestamp TIMESTAMP)""")

def log_run(batch_id, layer, entity, parameter, load_type, start, status,
            read=0, ins=0, upd=0, quar=0, drift=None, err=None):
    """Append one audit row. Called for every file/table processed, success or failure."""
    end = datetime.datetime.now()
    schema = StructType([StructField(n, t) for n, t in [
        ("batch_id", S), ("layer", S), ("entity", S), ("parameter", S), ("load_type", S),
        ("start_time", TimestampType()), ("end_time", TimestampType()), ("status", S),
        ("rows_read", L), ("rows_inserted", L), ("rows_updated", L), ("rows_quarantined", L),
        ("drift_columns", S), ("error_message", S), ("load_timestamp", TimestampType())]])
    row = (batch_id, layer, entity, parameter, load_type, start, end, status,
           read, ins, upd, quar, drift, err, end)
    spark.createDataFrame([row], schema).write.mode("append").saveAsTable(T("pipeline_execution_logs"))

# =====================================================================
# 4. IDEMPOTENT UPSERT (MERGE INTO)
# =====================================================================
def merge_metrics(table):
    m = DeltaTable.forName(spark, table).history(1).select("operationMetrics").first()[0]
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))

def upsert(df, table, pk, update_condition=None):
    """MERGE df into table on pk. Creates the table (empty, same schema) on first use.
    Returns (rows_inserted, rows_updated)."""
    if not spark.catalog.tableExists(table):
        df.limit(0).write.format("delta").saveAsTable(table)
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in pk)
    (DeltaTable.forName(spark, table).alias("t").merge(df.alias("s"), cond)
        .whenMatchedUpdateAll(condition=update_condition)
        .whenNotMatchedInsertAll().execute())
    return merge_metrics(table)

print("01_common loaded")